In [1]:
import numpy as np
import librosa
import sounddevice as sd
from tensorflow.keras.models import load_model


# CONFIG
SR = 22050
DURATION = 1.0   # seconds per chunk
N_MELS = 128
MAX_LEN = 85     # same as training
THRESHOLD = 0.002  # your computed value


# LOAD MODEL

model = load_model("autoencoder.h5", compile=False)


# LOAD SILENCE (same as training)

def extract_spectrogram(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

silence_audio, _ = librosa.load("./ReaLISED_Dataset/silence.flac", sr=SR)
silence_spec = extract_spectrogram(silence_audio)

# ensure long enough
if silence_spec.shape[1] < MAX_LEN:
    pad = MAX_LEN - silence_spec.shape[1]
    silence_spec = np.pad(
        silence_spec,
        ((0,0),(0,pad)),
        constant_values=np.min(silence_spec)
    )

# NOISY SILENCE

def get_noisy_silence(width):
    slice_ = silence_spec[:, :width].copy()
    noise = np.random.normal(0, 0.01, slice_.shape)
    return slice_ + noise

# -----------------------------
# PREPROCESS
# -----------------------------
def preprocess_audio(audio):
    spec = extract_spectrogram(audio)

    if spec.shape[1] < MAX_LEN:
        pad_width = MAX_LEN - spec.shape[1]

        pad_left = np.random.randint(0, pad_width + 1)
        pad_right = pad_width - pad_left

        left_pad = get_noisy_silence(pad_left)
        right_pad = get_noisy_silence(pad_right)

        spec = np.concatenate([left_pad, spec, right_pad], axis=1)
    else:
        spec = spec[:, :MAX_LEN]

    # normalize
    spec = (spec - np.min(spec)) / (np.max(spec) - np.min(spec) + 1e-8)

    spec = spec[np.newaxis, ..., np.newaxis]
    return spec


# MAIN LOOP

print("Listening... Press Ctrl+C to stop")

while True:
    # record audio
    audio = sd.rec(int(SR * DURATION), samplerate=SR, channels=1)
    sd.wait()

    audio = audio.flatten()

    # preprocess
    spec = preprocess_audio(audio)

    # predict
    recon = model.predict(spec, verbose=0)

    # error
    error = np.mean((spec - recon) ** 2)

    # decision
    if error > THRESHOLD:
        print(f"🚨 ANOMALY detected | error: {error:.5f}")
    else:
        print(f"✅ normal | error: {error:.5f}")

2026-04-18 20:12:07.970705: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 20:12:07.993102: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-18 20:12:08.191113: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-18 20:12:08.462886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776523328.643914    3822 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776523328.68

🎤 Listening... Press Ctrl+C to stop
🚨 ANOMALY detected | error: 0.00210
✅ normal | error: 0.00126
✅ normal | error: 0.00120
✅ normal | error: 0.00144
✅ normal | error: 0.00126
✅ normal | error: 0.00127
✅ normal | error: 0.00130
✅ normal | error: 0.00131
✅ normal | error: 0.00135
✅ normal | error: 0.00128
✅ normal | error: 0.00162
✅ normal | error: 0.00138
✅ normal | error: 0.00134
✅ normal | error: 0.00155
✅ normal | error: 0.00171
✅ normal | error: 0.00124
✅ normal | error: 0.00135
✅ normal | error: 0.00138
✅ normal | error: 0.00151
✅ normal | error: 0.00118
✅ normal | error: 0.00124
✅ normal | error: 0.00121
✅ normal | error: 0.00123
✅ normal | error: 0.00118
✅ normal | error: 0.00126
✅ normal | error: 0.00143
✅ normal | error: 0.00137
✅ normal | error: 0.00117
✅ normal | error: 0.00120
✅ normal | error: 0.00176
✅ normal | error: 0.00118
✅ normal | error: 0.00127
✅ normal | error: 0.00139


KeyboardInterrupt: 